In [1]:
pip install torch transformers datasets

In [2]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
from transformers import Trainer, TrainingArguments
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [3]:
from google.colab import files
uploaded = files.upload()

Saving data.txt to data (1).txt


In [4]:
!echo "Artificial Intelligence is transforming industries." > data.txt
!echo "Data Science helps in decision making." >> data.txt
!echo "GPT-2 can generate human-like text." >> data.txt
!echo "Machine learning models learn from data." >> data.txt


In [5]:
!cat data.txt


Artificial Intelligence is transforming industries.
Data Science helps in decision making.
GPT-2 can generate human-like text.
Machine learning models learn from data.


In [6]:
from datasets import load_dataset
dataset = load_dataset("text", data_files={"train": "data.txt"})

Generating train split: 0 examples [00:00, ? examples/s]

In [7]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token


def tokenize_function(examples):
    tokens = tokenizer(examples["text"], truncation=True, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()   # labels = input_ids
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [8]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Embedding(50257, 768)

In [9]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=100,
)


In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
)

trainer.train()


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2, training_loss=7.440594673156738, metrics={'train_runtime': 92.214, 'train_samples_per_second': 0.043, 'train_steps_per_second': 0.022, 'total_flos': 2090336256000.0, 'train_loss': 7.440594673156738, 'epoch': 1.0})

In [15]:
prompt = "Artificial Intelligence is"
inputs = tokenizer.encode(prompt, return_tensors="pt")

outputs = model.generate(
    inputs,
    max_length=30,       # किती लांब text हवा
    temperature=0.3,      # randomness control
    top_p=0.12,            # nucleus sampling
    do_sample=True        # random sampling enable
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Artificial Intelligence is a new technology that is being developed by the AI community.

The AI community is a community of the AI community is a


In [16]:
trainer.save_model("./my_gpt2_model")
tokenizer.save_pretrained("./my_gpt2_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_gpt2_model/tokenizer_config.json', './my_gpt2_model/tokenizer.json')

In [17]:
# for reusing
from transformers import GPT2LMHeadModel, GPT2Tokenizer
model = GPT2LMHeadModel.from_pretrained("./my_gpt2_model")
tokenizer = GPT2Tokenizer.from_pretrained("./my_gpt2_model")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]